In [1]:
!pip install scikit-surprise


In [2]:
!pip install "numpy<2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.36.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.2

In [1]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy


In [2]:
data = {
    'user_id': [1,1,1,2,2,3,3,4,4,5,5],
    'item_id': ['Movie1','Movie2','Movie3','Movie1','Movie3',
                'Movie2','Movie3','Movie1','Movie2','Movie3','Movie4'],
    'rating': [5,4,5,4,3,2,5,5,4,4,5]
}

df = pd.DataFrame(data)
df


,user_id,item_id,rating
0,1,Movie1,5
1,1,Movie2,4
2,1,Movie3,5
3,2,Movie1,4
4,2,Movie3,3
5,3,Movie2,2
6,3,Movie3,5
7,4,Movie1,5
8,4,Movie2,4
9,5,Movie3,4


In [3]:
reader = Reader(rating_scale=(1,5))
dataset = Dataset.load_from_df(df[['user_id','item_id','rating']], reader)


In [4]:
trainset, testset = train_test_split(dataset, test_size=0.2, random_state=42)


In [5]:
model = SVD()
model.fit(trainset)


In [6]:
predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

rmse, mae


RMSE: 0.9043
MAE:  0.7692


(0.9042830200107876, 0.7692128649082494)

In [7]:
def recommend_items(user_id, df, model, top_n=3):
    items = df['item_id'].unique()
    rated_items = df[df['user_id'] == user_id]['item_id'].tolist()

    recommendations = []
    for item in items:
        if item not in rated_items:
            pred = model.predict(user_id, item)
            recommendations.append((item, pred.est))

    recommendations.sort(key=lambda x: x[1], reverse=True)
    return recommendations[:top_n]


In [9]:
user_id = 1
recommendations = recommend_items(user_id, df, model)

print(f"Top Recommendations for User {user_id}:")
for item, rating in recommendations:
    print(f"{item} → Predicted Rating: {rating:.2f}")


Top Recommendations for User 1:
Movie4 → Predicted Rating: 4.02
